In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Execute batch preprocessing scripts sequentially for all months/quarters in 2023 and 2024
!python "/content/drive/MyDrive/Colab Notebooks/DataPreparationJAN2023.py"
!python "/content/drive/MyDrive/Colab Notebooks/DataPreparationQ12023.py"
!python "/content/drive/MyDrive/Colab Notebooks/DataPreparationQ22023.py"
!python "/content/drive/MyDrive/Colab Notebooks/DataPreparationQ32023.py"
!python "/content/drive/MyDrive/Colab Notebooks/DataPreparationQ42023.py"
!python "/content/drive/MyDrive/Colab Notebooks/DataPreparationJAN2024.py"
!python "/content/drive/MyDrive/Colab Notebooks/DataPreparationQ12024.py"
!python "/content/drive/MyDrive/Colab Notebooks/DataPreparationQ22024.py"
!python "/content/drive/MyDrive/Colab Notebooks/DataPreparationQ32024.py"
!python "/content/drive/MyDrive/Colab Notebooks/DataPreparationQ42024.py"

import dask.dataframe as dd
import matplotlib.pyplot as plt

# Load the preprocessed 2023 January and February-March (Q1.1) lazy Dask DataFrames
df_2023JAN = dd.read_parquet("drive/MyDrive/BT_Florian_2026/2023JAN_tripdata.parquet")
df_2023Q1 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/2023Q1.1_tripdata.parquet")

# Load the preprocessed 2024 January and February-March DataFrames
df_2024JAN = dd.read_parquet("drive/MyDrive/BT_Florian_2026/2024JAN_tripdata.parquet")
df_2024Q1 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/2024Q1.1_tripdata.parquet")

# Concatenate the 2023 components and trigger computation to materialize as a Pandas DataFrame
df_2023Q1 = dd.concat([df_2023JAN, df_2023Q1])
df_2023Q1 = df_2023Q1.compute()

# Concatenate the 2024 components and trigger computation to materialize as a Pandas DataFrame
df_2024Q1 = dd.concat([df_2024JAN, df_2024Q1])
df_2024Q1 = df_2024Q1.compute()

# Export the fully materialized Pandas DataFrames back to disk as unified Q1 Parquet files
df_2023Q1.to_parquet("drive/MyDrive/BT_Florian_2026/2023Q1_tripdata.parquet")
df_2024Q1.to_parquet("drive/MyDrive/BT_Florian_2026/2024Q1_tripdata.parquet")

In [ ]:
import pandas as pd
import dask.dataframe as dd

# Load the preprocessed quarterly and monthly trip datasets for 2023
df_2023JAN = dd.read_parquet("drive/MyDrive/BT_Florian_2026/Quarterly Tripdata/2023JAN_tripdata.parquet")
df_2023Q1 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/Quarterly Tripdata/2023Q1.1_tripdata.parquet")
df_2023Q2 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/Quarterly Tripdata/2023Q2_tripdata.parquet")
df_2023Q3 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/Quarterly Tripdata/2023Q3_tripdata.parquet")
df_2023Q4 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/Quarterly Tripdata/2023Q4_tripdata.parquet")

# Load the preprocessed quarterly and monthly trip datasets for 2024
df_2024JAN = dd.read_parquet("drive/MyDrive/BT_Florian_2026/Quarterly Tripdata/2024JAN_tripdata.parquet")
df_2024Q1 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/Quarterly Tripdata/2024Q1.1_tripdata.parquet")
df_2024Q2 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/Quarterly Tripdata/2024Q2_tripdata.parquet")
df_2024Q3 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/Quarterly Tripdata/2024Q3_tripdata.parquet")
df_2024Q4 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/Quarterly Tripdata/2024Q4_tripdata.parquet")

# Calculate the total number of records across the entire dataset by summing individual lengths
Rows = len(df_2023JAN) + len(df_2023Q1) + len(df_2023Q2) + len(df_2023Q3) + len(df_2023Q4) + len(df_2024JAN) + len(df_2024Q1) + len(df_2024Q2) + len(df_2024Q3) + len(df_2024Q4)

# Output the aggregate row count
print(Rows)

552887692


Downsizing the dataset:

In [ ]:
# 2023 Q1: Extract hourly aggregation keys, flag FHV trips, and calculate time durations
df_2023Q1["PUhour"] = dd.to_datetime(df_2023Q1["PUDatetime"]).dt.floor("h")
df_2023Q1["DOhour"] = dd.to_datetime(df_2023Q1["DODatetime"]).dt.floor("h")
df_2023Q1["fhvhv_percentage"] = df_2023Q1["taxi_type"] == "for-hire vehicle"
df_2023Q1["request2PU"] = df_2023Q1["PUDatetime"] - df_2023Q1["request_datetime"]
df_2023Q1["pickup2dropoff"] = df_2023Q1["DODatetime"] - df_2023Q1["PUDatetime"]

# 2023 Q2: Apply identical feature engineering
df_2023Q2["PUhour"] = dd.to_datetime(df_2023Q2["PUDatetime"]).dt.floor("h")
df_2023Q2["DOhour"] = dd.to_datetime(df_2023Q2["DODatetime"]).dt.floor("h")
df_2023Q2["fhvhv_percentage"] = df_2023Q2["taxi_type"] == "for-hire vehicle"
df_2023Q2["request2PU"] = df_2023Q2["PUDatetime"] - df_2023Q2["request_datetime"]
df_2023Q2["pickup2dropoff"] = df_2023Q2["DODatetime"] - df_2023Q2["PUDatetime"]

# 2023 Q3: Apply identical feature engineering
df_2023Q3["PUhour"] = dd.to_datetime(df_2023Q3["PUDatetime"]).dt.floor("h")
df_2023Q3["DOhour"] = dd.to_datetime(df_2023Q3["DODatetime"]).dt.floor("h")
df_2023Q3["fhvhv_percentage"] = df_2023Q3["taxi_type"] == "for-hire vehicle"
df_2023Q3["request2PU"] = df_2023Q3["PUDatetime"] - df_2023Q3["request_datetime"]
df_2023Q3["pickup2dropoff"] = df_2023Q3["DODatetime"] - df_2023Q3["PUDatetime"]

# 2023 Q4: Apply identical feature engineering
df_2023Q4["PUhour"] = dd.to_datetime(df_2023Q4["PUDatetime"]).dt.floor("h")
df_2023Q4["DOhour"] = dd.to_datetime(df_2023Q4["DODatetime"]).dt.floor("h")
df_2023Q4["fhvhv_percentage"] = df_2023Q4["taxi_type"] == "for-hire vehicle"
df_2023Q4["request2PU"] = df_2023Q4["PUDatetime"] - df_2023Q4["request_datetime"]
df_2023Q4["pickup2dropoff"] = df_2023Q4["DODatetime"] - df_2023Q4["PUDatetime"]


# 2024 Q1: Apply identical feature engineering
df_2024Q1["PUhour"] = dd.to_datetime(df_2024Q1["PUDatetime"]).dt.floor("h")
df_2024Q1["DOhour"] = dd.to_datetime(df_2024Q1["DODatetime"]).dt.floor("h")
df_2024Q1["fhvhv_percentage"] = df_2024Q1["taxi_type"] == "for-hire vehicle"
df_2024Q1["request2PU"] = df_2024Q1["PUDatetime"] - df_2024Q1["request_datetime"]
df_2024Q1["pickup2dropoff"] = df_2024Q1["DODatetime"] - df_2024Q1["PUDatetime"]

# 2024 Q2: Apply identical feature engineering
df_2024Q2["PUhour"] = dd.to_datetime(df_2024Q2["PUDatetime"]).dt.floor("h")
df_2024Q2["DOhour"] = dd.to_datetime(df_2024Q2["DODatetime"]).dt.floor("h")
df_2024Q2["fhvhv_percentage"] = df_2024Q2["taxi_type"] == "for-hire vehicle"
df_2024Q2["request2PU"] = df_2024Q2["PUDatetime"] - df_2024Q2["request_datetime"]
df_2024Q2["pickup2dropoff"] = df_2024Q2["DODatetime"] - df_2024Q2["PUDatetime"]

# 2024 Q3: Apply identical feature engineering
df_2024Q3["PUhour"] = dd.to_datetime(df_2024Q3["PUDatetime"]).dt.floor("h")
df_2024Q3["DOhour"] = dd.to_datetime(df_2024Q3["DODatetime"]).dt.floor("h")
df_2024Q3["fhvhv_percentage"] = df_2024Q3["taxi_type"] == "for-hire vehicle"
df_2024Q3["request2PU"] = df_2024Q3["PUDatetime"] - df_2024Q3["request_datetime"]
df_2024Q3["pickup2dropoff"] = df_2024Q3["DODatetime"] - df_2024Q3["PUDatetime"]

# 2024 Q4: Apply identical feature engineering
df_2024Q4["PUhour"] = dd.to_datetime(df_2024Q4["PUDatetime"]).dt.floor("h")
df_2024Q4["DOhour"] = dd.to_datetime(df_2024Q4["DODatetime"]).dt.floor("h")
df_2024Q4["fhvhv_percentage"] = df_2024Q4["taxi_type"] == "for-hire vehicle"
df_2024Q4["request2PU"] = df_2024Q4["PUDatetime"] - df_2024Q4["request_datetime"]
df_2024Q4["pickup2dropoff"] = df_2024Q4["DODatetime"] - df_2024Q4["PUDatetime"]



In [ ]:
# Aggregate 2023 Q1 data by pickup hour and location, calculating total trip volume and average metrics
df_PUs = df_2023Q1.groupby(["PUhour","PULocationID"]).agg({
    "PUDatetime":"count",
    "trip_distance":"mean",
    "fare_amount":"mean",
    "total_amount":"mean",
    "fhvhv_percentage":"mean",
    "tip_amount":"mean",
    "request2PU":"mean"
}).reset_index()

# Materialize the lazy Dask dataframe into a Pandas dataframe in memory
df_PUs = df_PUs.compute()

# Rename the count column for clarity and convert timedelta objects to numeric seconds
df_PUs = df_PUs.rename(columns={"PUDatetime":"trip_count"})
df_PUs["request2PU"] = df_PUs["request2PU"].dt.total_seconds()


import holidays
# Fetch NY state holidays for 2023 and format as datetime
holidays = holidays.US(subdiv="NY", years=2023)
holiday_dates = pd.to_datetime(list(holidays.keys()))

# Truncate the pickup hour to the day level to check if it falls on a holiday
PUDates = df_PUs["PUhour"].dt.floor("D")
df_PUs["is_holiday"] = PUDates.isin(holiday_dates)

# Extract day of the week (0=Monday, 6=Sunday)
df_PUs["DayOfWeek"] = df_PUs["PUhour"].dt.dayofweek

# Save the finalized Q1 hourly dataset to disk
df_PUs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2023Q1.parquet")


# Repeat the exact same aggregation and feature engineering process for 2023 Q2
df_PUs = df_2023Q2.groupby(["PUhour","PULocationID"]).agg({
    "PUDatetime":"count",
    "trip_distance":"mean",
    "fare_amount":"mean",
    "total_amount":"mean",
    "fhvhv_percentage":"mean",
    "tip_amount":"mean",
    "request2PU":"mean"
}).reset_index()

df_PUs = df_PUs.compute()

df_PUs = df_PUs.rename(columns={"PUDatetime":"trip_count"})
df_PUs["request2PU"] = df_PUs["request2PU"].dt.total_seconds()


import holidays
holidays = holidays.US(subdiv="NY", years=2023)
holiday_dates = pd.to_datetime(list(holidays.keys()))

PUDates = df_PUs["PUhour"].dt.floor("D")
df_PUs["is_holiday"] = PUDates.isin(holiday_dates)

df_PUs["DayOfWeek"] = df_PUs["PUhour"].dt.dayofweek

df_PUs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2023Q2.parquet")


# Repeat the exact same aggregation and feature engineering process for 2023 Q3
df_PUs = df_2023Q3.groupby(["PUhour","PULocationID"]).agg({
    "PUDatetime":"count",
    "trip_distance":"mean",
    "fare_amount":"mean",
    "total_amount":"mean",
    "fhvhv_percentage":"mean",
    "tip_amount":"mean",
    "request2PU":"mean"
}).reset_index()

df_PUs = df_PUs.compute()

df_PUs = df_PUs.rename(columns={"PUDatetime":"trip_count"})
df_PUs["request2PU"] = df_PUs["request2PU"].dt.total_seconds()


import holidays
holidays = holidays.US(subdiv="NY", years=2023)
holiday_dates = pd.to_datetime(list(holidays.keys()))

PUDates = df_PUs["PUhour"].dt.floor("D")
df_PUs["is_holiday"] = PUDates.isin(holiday_dates)

df_PUs["DayOfWeek"] = df_PUs["PUhour"].dt.dayofweek

df_PUs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2023Q3.parquet")


# Repeat the exact same aggregation and feature engineering process for 2023 Q4
df_PUs = df_2023Q4.groupby(["PUhour","PULocationID"]).agg({
    "PUDatetime":"count",
    "trip_distance":"mean",
    "fare_amount":"mean",
    "total_amount":"mean",
    "fhvhv_percentage":"mean",
    "tip_amount":"mean",
    "request2PU":"mean"
}).reset_index()

df_PUs = df_PUs.compute()

df_PUs = df_PUs.rename(columns={"PUDatetime":"trip_count"})
df_PUs["request2PU"] = df_PUs["request2PU"].dt.total_seconds()


import holidays
holidays = holidays.US(subdiv="NY", years=2023)
holiday_dates = pd.to_datetime(list(holidays.keys()))

PUDates = df_PUs["PUhour"].dt.floor("D")
df_PUs["is_holiday"] = PUDates.isin(holiday_dates)

df_PUs["DayOfWeek"] = df_PUs["PUhour"].dt.dayofweek

df_PUs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2023Q4.parquet")


In [ ]:
# Aggregate 2024 Q1 pick-up data by hour and location, calculating trip volume and average metrics
df_PUs = df_2024Q1.groupby(["PUhour","PULocationID"]).agg({
    "PUDatetime":"count",
    "trip_distance":"mean",
    "fare_amount":"mean",
    "total_amount":"mean",
    "fhvhv_percentage":"mean",
    "tip_amount":"mean",
    "request2PU":"mean"
}).reset_index()

# Materialize the lazy Dask dataframe into memory as a Pandas dataframe
df_PUs = df_PUs.compute()

# Rename the count column for clarity and convert timedelta objects to numeric seconds
df_PUs = df_PUs.rename(columns={"PUDatetime":"trip_count"})
df_PUs["request2PU"] = df_PUs["request2PU"].dt.total_seconds()


import holidays
# Fetch NY state holidays for 2024 and format as datetime
holidays = holidays.US(subdiv="NY", years=2024)
holiday_dates = pd.to_datetime(list(holidays.keys()))

# Truncate the pickup hour to the day level to check if it falls on a holiday
PUDates = df_PUs["PUhour"].dt.floor("D")
df_PUs["is_holiday"] = PUDates.isin(holiday_dates)

# Extract day of the week (0=Monday, 6=Sunday)
df_PUs["DayOfWeek"] = df_PUs["PUhour"].dt.dayofweek

# Save the finalized Q1 hourly dataset to disk
df_PUs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2024Q1.parquet")


# Repeat the exact same aggregation and feature engineering process for 2024 Q2
df_PUs = df_2024Q2.groupby(["PUhour","PULocationID"]).agg({
    "PUDatetime":"count",
    "trip_distance":"mean",
    "fare_amount":"mean",
    "total_amount":"mean",
    "fhvhv_percentage":"mean",
    "tip_amount":"mean",
    "request2PU":"mean"
}).reset_index()

df_PUs = df_PUs.compute()

df_PUs = df_PUs.rename(columns={"PUDatetime":"trip_count"})
df_PUs["request2PU"] = df_PUs["request2PU"].dt.total_seconds()


import holidays
holidays = holidays.US(subdiv="NY", years=2024)
holiday_dates = pd.to_datetime(list(holidays.keys()))

PUDates = df_PUs["PUhour"].dt.floor("D")
df_PUs["is_holiday"] = PUDates.isin(holiday_dates)

df_PUs["DayOfWeek"] = df_PUs["PUhour"].dt.dayofweek

df_PUs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2024Q2.parquet")


# Repeat the exact same aggregation and feature engineering process for 2024 Q3
df_PUs = df_2024Q3.groupby(["PUhour","PULocationID"]).agg({
    "PUDatetime":"count",
    "trip_distance":"mean",
    "fare_amount":"mean",
    "total_amount":"mean",
    "fhvhv_percentage":"mean",
    "tip_amount":"mean",
    "request2PU":"mean"
}).reset_index()

df_PUs = df_PUs.compute()

df_PUs = df_PUs.rename(columns={"PUDatetime":"trip_count"})
df_PUs["request2PU"] = df_PUs["request2PU"].dt.total_seconds()


import holidays
holidays = holidays.US(subdiv="NY", years=2024)
holiday_dates = pd.to_datetime(list(holidays.keys()))

PUDates = df_PUs["PUhour"].dt.floor("D")
df_PUs["is_holiday"] = PUDates.isin(holiday_dates)

df_PUs["DayOfWeek"] = df_PUs["PUhour"].dt.dayofweek

df_PUs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2024Q3.parquet")


# Repeat the exact same aggregation and feature engineering process for 2024 Q4
df_PUs = df_2024Q4.groupby(["PUhour","PULocationID"]).agg({
    "PUDatetime":"count",
    "trip_distance":"mean",
    "fare_amount":"mean",
    "total_amount":"mean",
    "fhvhv_percentage":"mean",
    "tip_amount":"mean",
    "request2PU":"mean"
}).reset_index()

df_PUs = df_PUs.compute()

df_PUs = df_PUs.rename(columns={"PUDatetime":"trip_count"})
df_PUs["request2PU"] = df_PUs["request2PU"].dt.total_seconds()


import holidays
holidays = holidays.US(subdiv="NY", years=2024)
holiday_dates = pd.to_datetime(list(holidays.keys()))

PUDates = df_PUs["PUhour"].dt.floor("D")
df_PUs["is_holiday"] = PUDates.isin(holiday_dates)

df_PUs["DayOfWeek"] = df_PUs["PUhour"].dt.dayofweek

df_PUs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2024Q4.parquet")

In [ ]:
# Aggregate 2023 Q1 drop-off data by hour and destination location, calculating trip volume and average metrics
df_DOs = df_2023Q1.groupby(["DOhour","DOLocationID"]).agg({
    "DODatetime":"count",
    "trip_distance":"mean",
    "fare_amount":"mean",
    "total_amount":"mean",
    "fhvhv_percentage":"mean",
    "tip_amount":"mean",
    "pickup2dropoff":"mean"
}).reset_index()

# Materialize the lazy Dask dataframe into memory as a Pandas dataframe
df_DOs = df_DOs.compute()

# Rename the count column to represent total trip volume and convert timedelta duration to numeric seconds
df_DOs = df_DOs.rename(columns={"DODatetime":"trip_count"})
df_DOs["pickup2dropoff"] = df_DOs["pickup2dropoff"].dt.total_seconds()


import holidays
# Fetch NY state holidays for 2023 and format them as datetime objects
holidays = holidays.US(subdiv="NY", years=2023)
holiday_dates = pd.to_datetime(list(holidays.keys()))

# Truncate the drop-off hour to the day level to check if the date matches a holiday
DODates = df_DOs["DOhour"].dt.floor("D")
df_DOs["is_holiday"] = DODates.isin(holiday_dates)

# Extract the day of the week as an integer (0=Monday, 6=Sunday)
df_DOs["DayOfWeek"] = df_DOs["DOhour"].dt.dayofweek

# Save the finalized Q1 hourly drop-off dataset to disk in Parquet format
df_DOs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2023Q1.parquet")




# Repeat the exact same aggregation and feature engineering process for 2023 Q2 drop-offs
df_DOs = df_2023Q2.groupby(["DOhour","DOLocationID"]).agg({
    "DODatetime":"count",
    "trip_distance":"mean",
    "fare_amount":"mean",
    "total_amount":"mean",
    "fhvhv_percentage":"mean",
    "tip_amount":"mean",
    "pickup2dropoff":"mean"
}).reset_index()

df_DOs = df_DOs.compute()

df_DOs = df_DOs.rename(columns={"DODatetime":"trip_count"})
df_DOs["pickup2dropoff"] = df_DOs["pickup2dropoff"].dt.total_seconds()


import holidays
holidays = holidays.US(subdiv="NY", years=2023)
holiday_dates = pd.to_datetime(list(holidays.keys()))

DODates = df_DOs["DOhour"].dt.floor("D")
df_DOs["is_holiday"] = DODates.isin(holiday_dates)

df_DOs["DayOfWeek"] = df_DOs["DOhour"].dt.dayofweek

df_DOs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2023Q2.parquet")




# Repeat the exact same aggregation and feature engineering process for 2023 Q3 drop-offs
df_DOs = df_2023Q3.groupby(["DOhour","DOLocationID"]).agg({
    "DODatetime":"count",
    "trip_distance":"mean",
    "fare_amount":"mean",
    "total_amount":"mean",
    "fhvhv_percentage":"mean",
    "tip_amount":"mean",
    "pickup2dropoff":"mean"
}).reset_index()

df_DOs = df_DOs.compute()

df_DOs = df_DOs.rename(columns={"DODatetime":"trip_count"})
df_DOs["pickup2dropoff"] = df_DOs["pickup2dropoff"].dt.total_seconds()


import holidays
holidays = holidays.US(subdiv="NY", years=2023)
holiday_dates = pd.to_datetime(list(holidays.keys()))

DODates = df_DOs["DOhour"].dt.floor("D")
df_DOs["is_holiday"] = DODates.isin(holiday_dates)

df_DOs["DayOfWeek"] = df_DOs["DOhour"].dt.dayofweek

df_DOs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2023Q3.parquet")




# Repeat the exact same aggregation and feature engineering process for 2023 Q4 drop-offs
df_DOs = df_2023Q4.groupby(["DOhour","DOLocationID"]).agg({
    "DODatetime":"count",
    "trip_distance":"mean",
    "fare_amount":"mean",
    "total_amount":"mean",
    "fhvhv_percentage":"mean",
    "tip_amount":"mean",
    "pickup2dropoff":"mean"
}).reset_index()

df_DOs = df_DOs.compute()

df_DOs = df_DOs.rename(columns={"DODatetime":"trip_count"})
df_DOs["pickup2dropoff"] = df_DOs["pickup2dropoff"].dt.total_seconds()


import holidays
holidays = holidays.US(subdiv="NY", years=2023)
holiday_dates = pd.to_datetime(list(holidays.keys()))

DODates = df_DOs["DOhour"].dt.floor("D")
df_DOs["is_holiday"] = DODates.isin(holiday_dates)

df_DOs["DayOfWeek"] = df_DOs["DOhour"].dt.dayofweek

df_DOs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2023Q4.parquet")

In [ ]:
# Aggregate 2024 Q1 drop-off data by hour and destination location, computing total trips and average metrics
df_DOs = df_2024Q1.groupby(["DOhour","DOLocationID"]).agg({
    "DODatetime":"count",
    "trip_distance":"mean",
    "fare_amount":"mean",
    "total_amount":"mean",
    "fhvhv_percentage":"mean",
    "tip_amount":"mean",
    "pickup2dropoff":"mean"
}).reset_index()

# Materialize the lazy Dask dataframe into memory as a Pandas dataframe
df_DOs = df_DOs.compute()

# Rename the count column to represent trip volume and convert the timedelta duration to numeric seconds
df_DOs = df_DOs.rename(columns={"DODatetime":"trip_count"})
df_DOs["pickup2dropoff"] = df_DOs["pickup2dropoff"].dt.total_seconds()


import holidays
# Fetch NY state holidays for 2024 and format them as datetime objects
holidays = holidays.US(subdiv="NY", years=2024)
holiday_dates = pd.to_datetime(list(holidays.keys()))

# Truncate the drop-off hour to the day level to evaluate whether it falls on a holiday
DODates = df_DOs["DOhour"].dt.floor("D")
df_DOs["is_holiday"] = DODates.isin(holiday_dates)

# Extract the day of the week as an integer (0=Monday, 6=Sunday)
df_DOs["DayOfWeek"] = df_DOs["DOhour"].dt.dayofweek

# Save the finalized 2024 Q1 hourly drop-off dataset to disk in Parquet format
df_DOs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2024Q1.parquet")




# Repeat the exact same aggregation and feature engineering process for 2024 Q2 drop-offs
df_DOs = df_2024Q2.groupby(["DOhour","DOLocationID"]).agg({
    "DODatetime":"count",
    "trip_distance":"mean",
    "fare_amount":"mean",
    "total_amount":"mean",
    "fhvhv_percentage":"mean",
    "tip_amount":"mean",
    "pickup2dropoff":"mean"
}).reset_index()

df_DOs = df_DOs.compute()

df_DOs = df_DOs.rename(columns={"DODatetime":"trip_count"})
df_DOs["pickup2dropoff"] = df_DOs["pickup2dropoff"].dt.total_seconds()


import holidays
holidays = holidays.US(subdiv="NY", years=2024)
holiday_dates = pd.to_datetime(list(holidays.keys()))

DODates = df_DOs["DOhour"].dt.floor("D")
df_DOs["is_holiday"] = DODates.isin(holiday_dates)

df_DOs["DayOfWeek"] = df_DOs["DOhour"].dt.dayofweek

df_DOs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2024Q2.parquet")




# Repeat the exact same aggregation and feature engineering process for 2024 Q3 drop-offs
df_DOs = df_2024Q3.groupby(["DOhour","DOLocationID"]).agg({
    "DODatetime":"count",
    "trip_distance":"mean",
    "fare_amount":"mean",
    "total_amount":"mean",
    "fhvhv_percentage":"mean",
    "tip_amount":"mean",
    "pickup2dropoff":"mean"
}).reset_index()

df_DOs = df_DOs.compute()

df_DOs = df_DOs.rename(columns={"DODatetime":"trip_count"})
df_DOs["pickup2dropoff"] = df_DOs["pickup2dropoff"].dt.total_seconds()


import holidays
holidays = holidays.US(subdiv="NY", years=2024)
holiday_dates = pd.to_datetime(list(holidays.keys()))

DODates = df_DOs["DOhour"].dt.floor("D")
df_DOs["is_holiday"] = DODates.isin(holiday_dates)

df_DOs["DayOfWeek"] = df_DOs["DOhour"].dt.dayofweek

df_DOs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2024Q3.parquet")




# Repeat the exact same aggregation and feature engineering process for 2024 Q4 drop-offs
df_DOs = df_2024Q4.groupby(["DOhour","DOLocationID"]).agg({
    "DODatetime":"count",
    "trip_distance":"mean",
    "fare_amount":"mean",
    "total_amount":"mean",
    "fhvhv_percentage":"mean",
    "tip_amount":"mean",
    "pickup2dropoff":"mean"
}).reset_index()

df_DOs = df_DOs.compute()

df_DOs = df_DOs.rename(columns={"DODatetime":"trip_count"})
df_DOs["pickup2dropoff"] = df_DOs["pickup2dropoff"].dt.total_seconds()


import holidays
holidays = holidays.US(subdiv="NY", years=2024)
holiday_dates = pd.to_datetime(list(holidays.keys()))

DODates = df_DOs["DOhour"].dt.floor("D")
df_DOs["is_holiday"] = DODates.isin(holiday_dates)

df_DOs["DayOfWeek"] = df_DOs["DOhour"].dt.dayofweek

df_DOs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2024Q4.parquet")

In [ ]:
# Load all hourly aggregated Pick-Up (PU) datasets for each quarter of 2023
df_PUsQ123 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2023Q1.parquet")
df_PUsQ223 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2023Q2.parquet")
df_PUsQ323 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2023Q3.parquet")
df_PUsQ423 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2023Q4.parquet")

# Load all hourly aggregated Pick-Up (PU) datasets for each quarter of 2024
df_PUsQ124 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2024Q1.parquet")
df_PUsQ224 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2024Q2.parquet")
df_PUsQ324 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2024Q3.parquet")
df_PUsQ424 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2024Q4.parquet")


# Load all hourly aggregated Drop-Off (DO) datasets for each quarter of 2023
df_DOsQ123 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2024Q1.parquet")
df_DOsQ223 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2024Q2.parquet")
df_DOsQ323 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2024Q3.parquet")
df_DOsQ423 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2024Q4.parquet")

# Load all hourly aggregated Drop-Off (DO) datasets for each quarter of 2024
df_DOsQ124 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2024Q1.parquet")
df_DOsQ224 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2024Q2.parquet")
df_DOsQ324 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2024Q3.parquet")
df_DOsQ424 = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2024Q4.parquet")


# Concatenate all eight quarterly Pick-Up dataframes into a single continuous Dask dataframe spanning 2023-2024
df_PUs = dd.concat([df_PUsQ123,df_PUsQ223,df_PUsQ323,df_PUsQ423,df_PUsQ124,df_PUsQ224,df_PUsQ324,df_PUsQ424])

# Concatenate all eight quarterly Drop-Off dataframes into a single continuous Dask dataframe spanning 2023-2024
df_DOs = dd.concat([df_DOsQ123,df_DOsQ223,df_DOsQ323,df_DOsQ423,df_DOsQ124,df_DOsQ224,df_DOsQ324,df_DOsQ424])


# Export the consolidated Pick-Up and Drop-Off datasets to master Parquet files for downstream normalization and modeling
df_PUs = df_PUs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2023-24RAW.parquet")
df_DOs = df_DOs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2023-24RAW.parquet")

Adding weather data:

In [ ]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

In [ ]:
# Code for this cell was taken form: https://open-meteo.com/en/docs/historical-weather-api?latitude=40.7143&longitude=-74.006&start_date=2023-01-01&end_date=2024-12-31
# Accessed on 2026-08-28

import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 40.7143,
	"longitude": -74.006,
	"start_date": "2022-12-31",
	"end_date": "2024-12-31",
	"hourly": ["temperature_2m", "rain", "snow_depth", "apparent_temperature", "snowfall", "precipitation"],
  "timezone": "America/New_York",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_rain = hourly.Variables(1).ValuesAsNumpy()
hourly_snow_depth = hourly.Variables(2).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(3).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(4).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(5).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["rain"] = hourly_rain
hourly_data["snow_depth"] = hourly_snow_depth
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["snowfall"] = hourly_snowfall
hourly_data["precipitation"] = hourly_precipitation


hourly_dataframe = pd.DataFrame(data = hourly_data)
hourly_dataframe["date"] = hourly_dataframe["date"].dt.tz_localize(None)
print("\nHourly data\n", hourly_dataframe)

Coordinates: 40.738136291503906°N -74.04254150390625°E
Elevation: 51.0 m asl
Timezone difference to GMT+0: -14400s

Hourly data
                      date  temperature_2m  rain  snow_depth  \
0     2022-12-31 04:00:00            6.00   0.0         0.0   
1     2022-12-31 05:00:00            6.20   0.0         0.0   
2     2022-12-31 06:00:00            6.50   0.0         0.0   
3     2022-12-31 07:00:00            6.25   0.0         0.0   
4     2022-12-31 08:00:00            6.30   0.0         0.0   
...                   ...             ...   ...         ...   
17563 2024-12-31 23:00:00            7.85   0.0         0.0   
17564 2025-01-01 00:00:00            8.20   0.0         0.0   
17565 2025-01-01 01:00:00            8.10   0.0         0.0   
17566 2025-01-01 02:00:00            7.45   3.1         0.0   
17567 2025-01-01 03:00:00            7.45   2.8         0.0   

       apparent_temperature  snowfall  precipitation  
0                  3.288507       0.0            0.0  
1   

In [ ]:
import dask.dataframe as dd
import pandas as pd

# Load the concatenated 2023-2024 raw Pick-Up and Drop-Off Parquet files
dfPUs  = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2023-24RAW.parquet")
dfDOs  = dd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2023-24RAW.parquet")

# Ensure the Pick-Up hour column is properly cast to datetime for merging
dfPUs["PUhour"] = dd.to_datetime(dfPUs["PUhour"])

# Merge the Pick-Up data with the external hourly weather dataframe
dfPUs = dd.merge(
    dfPUs,
    hourly_dataframe,
    left_on="PUhour",
    right_on="date",
    how="left"
)

# Drop the redundant date column from the weather dataset after the merge
dfPUs = dfPUs.drop(columns=["date"])


# Ensure the Drop-Off hour column is properly cast to datetime for merging
dfDOs["DOhour"] = dd.to_datetime(dfDOs["DOhour"])

# Merge the Drop-Off data with the external hourly weather dataframe
dfDOs = dd.merge(
    dfDOs,
    hourly_dataframe,
    left_on="DOhour",
    right_on="date",
    how="left"
)

# Drop the redundant date column from the weather dataset after the merge
dfDOs = dfDOs.drop(columns=["date"])


# Convert boolean holiday flags into integer format (0 or 1) for modeling
dfPUs["is_holiday"] = dfPUs["is_holiday"].astype("int")
dfDOs["is_holiday"] = dfDOs["is_holiday"].astype("int")

# Extract granular temporal components from the Pick-Up datetime
dfPUs["year"] = dfPUs["PUhour"].dt.year
dfPUs["month"] = dfPUs["PUhour"].dt.month
dfPUs["day"] = dfPUs["PUhour"].dt.day
dfPUs["hour"] = dfPUs["PUhour"].dt.hour

# Extract granular temporal components from the Drop-Off datetime
dfDOs["year"] = dfDOs["DOhour"].dt.year
dfDOs["month"] = dfDOs["DOhour"].dt.month
dfDOs["day"] = dfDOs["DOhour"].dt.day
dfDOs["hour"] = dfDOs["DOhour"].dt.hour

# Cast Location IDs to categorical variables for memory efficiency and modeling purposes
dfPUs["PULocationID"] = dfPUs["PULocationID"].astype("category")
dfDOs["DOLocationID"] = dfDOs["DOLocationID"].astype("category")


# Materialize the lazy Dask DataFrames into in-memory Pandas DataFrames
dfPUs = dfPUs.compute()
dfDOs = dfDOs.compute()

# Enforce a structured column order combining core metrics, time features, and weather data
dfPUs = dfPUs.reindex(columns=["trip_count","PUhour","year","month","day","hour","DayOfWeek",
                               "is_holiday","PULocationID", "trip_distance","fare_amount","total_amount",
                               "fhvhv_percentage","tip_amount","request2PU",
                               "rain", "snow_depth", "apparent_temperature", "snowfall", "precipitation"])

dfDOs = dfDOs.reindex(columns=["trip_count","DOhour","year","month","day","hour","DayOfWeek",
                               "is_holiday","DOLocationID","trip_distance","fare_amount","total_amount",
                               "fhvhv_percentage","tip_amount","pickup2dropoff",
                               "rain", "snow_depth", "apparent_temperature", "snowfall", "precipitation"])

# Export the fully engineered, weather-augmented datasets to Parquet files
dfPUs  = dfPUs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2023-24WEATHER.parquet")
dfDOs  = dfDOs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2023-24WEATHER.parquet")


In [ ]:
import pandas as pd

# Load the weather-augmented Pick-Up and Drop-Off datasets into Pandas DataFrames
dfPUs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2023-24WEATHER.parquet")
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2023-24WEATHER.parquet")

# Drop the last 5 columns (the weather features) to create a baseline dataset without weather data
dfPUs = dfPUs.drop(columns=dfPUs.columns[-5:])
dfDOs = dfDOs.drop(columns=dfDOs.columns[-5:])

# Export the baseline datasets to new Parquet files
dfPUs  = dfPUs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2023-24.parquet")
dfDOs  = dfDOs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2023-24.parquet")

In [ ]:
import pandas as pd
import numpy as np

# Load the datasets containing both trip data and weather features
dfPUs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2023-24WEATHER.parquet")
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2023-24WEATHER.parquet")

# Apply cyclical encoding to the hour of the day (period = 24)
# This ensures models interpret 23:00 and 00:00 as continuous rather than distant values
dfPUs["hour_sin"] = np.sin(2 * np.pi * dfPUs["hour"] / 24)
dfPUs["hour_cos"] = np.cos(2 * np.pi * dfPUs["hour"] / 24)
dfDOs["hour_sin"] = np.sin(2 * np.pi * dfDOs["hour"] / 24)
dfDOs["hour_cos"] = np.cos(2 * np.pi * dfDOs["hour"] / 24)

# Apply cyclical encoding to the day of the week (period = 7)
dfPUs["DoW_sin"] = np.sin(2 * np.pi * dfPUs["DayOfWeek"] / 7)
dfPUs["DoW_cos"] = np.cos(2 * np.pi * dfPUs["DayOfWeek"] / 7)
dfDOs["DoW_sin"] = np.sin(2 * np.pi * dfDOs["DayOfWeek"] / 7)
dfDOs["DoW_cos"] = np.cos(2 * np.pi * dfDOs["DayOfWeek"] / 7)

# Apply cyclical encoding to the day of the month dynamically
# The period adjusts automatically based on the actual number of days in the given month (28, 29, 30, or 31)
dfPUs["day_sin"] = np.sin(2 * np.pi * dfPUs["day"] / dfPUs["PUhour"].dt.days_in_month)
dfPUs["day_cos"] = np.cos(2 * np.pi * dfPUs["day"] / dfPUs["PUhour"].dt.days_in_month)
dfDOs["day_sin"] = np.sin(2 * np.pi * dfDOs["day"] / dfDOs["DOhour"].dt.days_in_month)
dfDOs["day_cos"] = np.cos(2 * np.pi * dfDOs["day"] / dfDOs["DOhour"].dt.days_in_month)

# Apply cyclical encoding to the month of the year (period = 12)
dfPUs["month_sin"] = np.sin(2 * np.pi * dfPUs["month"] / 12)
dfPUs["month_cos"] = np.cos(2 * np.pi * dfPUs["month"] / 12)
dfDOs["month_sin"] = np.sin(2 * np.pi * dfDOs["month"] / 12)
dfDOs["month_cos"] = np.cos(2 * np.pi * dfDOs["month"] / 12)

# Drop the original linear time components
dfPUs = dfPUs.drop(columns=["hour","DayOfWeek","day","month"])
dfDOs = dfDOs.drop(columns=["hour","DayOfWeek","day","month"])

# Reorder the columns to establish a standardized schema placing the new cyclical features together
dfPUs = dfPUs.reindex(columns=["trip_count","PUhour","year","month_sin","month_cos","day_sin","day_cos",
                               "hour_sin","hour_cos","DoW_sin","DoW_cos",
                               "is_holiday","PULocationID", "trip_distance","fare_amount","total_amount",
                               "fhvhv_percentage","tip_amount","request2PU",
                               "rain","snow_depth","apparent_temperature","snowfall","precipitation"])

dfDOs = dfDOs.reindex(columns=["trip_count","DOhour","year","month_sin","month_cos","day_sin","day_cos",
                               "hour_sin","hour_cos","DoW_sin","DoW_cos",
                               "is_holiday","DOLocationID","trip_distance","fare_amount","total_amount",
                               "fhvhv_percentage","tip_amount","pickup2dropoff",
                               "rain","snow_depth","apparent_temperature","snowfall","precipitation"])

# Export the fully engineered datasets containing both weather and cyclical time features
dfPUs  = dfPUs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2023-24WEATHER_CYCLt.parquet")
dfDOs  = dfDOs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2023-24WEATHER_CYCLt.parquet")

In [ ]:
import pandas as pd

# Load the final comprehensive datasets containing cyclical time features and weather augmentations
dfPUs  = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24WEATHER_CYCLt.parquet")
dfDOs  = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24WEATHER_CYCLt.parquet")

# Debugging/Validation check: Count the number of grouped hourly records where the average trip distance exceeds 200 miles.
# Since these are aggregated means, even a small number here suggests extreme outliers in the raw data (or faulty clock entries).
print((dfPUs["trip_distance"] > 200).sum())
print((dfDOs["trip_distance"] > 200).sum())

# Strip the 5 weather columns off the ends of the dataframes to create a cyclical-only dataset
dfPUs = dfPUs.drop(columns=dfPUs.columns[-5:])
dfDOs = dfDOs.drop(columns=dfDOs.columns[-5:])

# Export the cyclical-only Pick-Up dataset to disk as a Parquet file
dfPUs  = dfPUs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyPUs2023-24CYCLt.parquet")
# Export the cyclical-only Drop-Off dataset to disk as a Parquet file
dfDOs  = dfDOs.to_parquet("drive/MyDrive/BT_Florian_2026/hourlyDOs2023-24CYCLt.parquet")

1222
1016


In [ ]:
import pandas as pd

# Load the finalized Pick-Up and Drop-Off datasets enriched with weather features from Parquet files
dfPUs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24WEATHER.parquet")
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24WEATHER.parquet")

# Output the total number of records in each dataframe to verify dataset scale and completeness
print(len(dfPUs))
print(len(dfDOs))

# Extract a 200-row sample from each dataset and export to CSV format for quick visual inspection
dfPUs = dfPUs.head(200).to_csv("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24WEATHER.csv")
dfDOs = dfDOs.head(200).to_csv("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24WEATHER.csv")

4446461
4480893


Add geographic coordnates to the taxi zones

In [ ]:
import pandas as pd
import geopandas as gpd

# Load the spatial shapefile containing NYC Taxi zone polygons
shapefile_url = "drive/MyDrive/BT_Florian_2026/taxi_zones/taxi_zones.shp"
zones = gpd.read_file(shapefile_url)

# Convert to a projected coordinate reference system (Web Mercator EPSG:3857)
# This projects the map flat, ensuring accurate geometric centroid calculations
# (AI generated)
zones_flat = zones.to_crs("EPSG:3857")

# Calculate the geographic center point of each taxi zone polygon
# (AI generated)
flat_centroids = zones_flat.geometry.centroid

# Convert the calculated centroids back to standard GPS coordinates (WGS 84 EPSG:4326)
# (AI generated)
gps_centroids = flat_centroids.to_crs("EPSG:4326")

# Extract the longitude (x) and latitude (y) to prepare for merging
# (AI generated)
zones["longitude"] = gps_centroids.x
zones["latitude"] = gps_centroids.y

# Isolate the mapping table of Location IDs to their respective GPS coordinates
# (AI generated)
zone_coords = zones[["LocationID", "longitude", "latitude"]]


# --- Dataset Variant 1: Baseline Features ---
# Load the baseline Pick-Up and Drop-Off datasets
dfPUs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24.parquet")
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24.parquet")

# Merge the spatial centroid coordinates based on the respective location IDs
dfPUs = dfPUs.merge(zone_coords, left_on="PULocationID", right_on="LocationID", how="left")
dfDOs = dfDOs.merge(zone_coords, left_on="DOLocationID", right_on="LocationID", how="left")

# Drop the duplicate LocationID column left over from the merge
dfPUs = dfPUs.drop(columns=["LocationID"])
dfDOs = dfDOs.drop(columns=["LocationID"])

# Overwrite the original files with the newly spatially-enriched datasets
dfPUs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24.parquet")
dfDOs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24.parquet")


# --- Dataset Variant 2: Weather Features Included ---
# Apply the exact same spatial join process to the weather-augmented datasets
dfPUs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24WEATHER.parquet")
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24WEATHER.parquet")

dfPUs = dfPUs.merge(zone_coords, left_on="PULocationID", right_on="LocationID", how="left")
dfDOs = dfDOs.merge(zone_coords, left_on="DOLocationID", right_on="LocationID", how="left")

dfPUs = dfPUs.drop(columns=["LocationID"])
dfDOs = dfDOs.drop(columns=["LocationID"])

dfPUs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24WEATHER.parquet")
dfDOs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24WEATHER.parquet")


# --- Dataset Variant 3: Weather and Cyclical Time Features Included ---
# Apply the exact same spatial join process to the comprehensive datasets
dfPUs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24WEATHER_CYCLt.parquet")
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24WEATHER_CYCLt.parquet")

dfPUs = dfPUs.merge(zone_coords, left_on="PULocationID", right_on="LocationID", how="left")
dfDOs = dfDOs.merge(zone_coords, left_on="DOLocationID", right_on="LocationID", how="left")

dfPUs = dfPUs.drop(columns=["LocationID"])
dfDOs = dfDOs.drop(columns=["LocationID"])

dfPUs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24WEATHER_CYCLt.parquet")
dfDOs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24WEATHER_CYCLt.parquet")


# --- Dataset Variant 4: Cyclical Time Features Included (No Weather) ---
# Apply the exact same spatial join process to the cyclical-only datasets
dfPUs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24CYCLt.parquet")
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24CYCLt.parquet")

dfPUs = dfPUs.merge(zone_coords, left_on="PULocationID", right_on="LocationID", how="left")
dfDOs = dfDOs.merge(zone_coords, left_on="DOLocationID", right_on="LocationID", how="left")

dfPUs = dfPUs.drop(columns=["LocationID"])
dfDOs = dfDOs.drop(columns=["LocationID"])

dfPUs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24CYCLt.parquet")
dfDOs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24CYCLt.parquet")

In [ ]:
import pandas as pd

# Load the baseline Pick-Up and Drop-Off datasets
dfPUs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24.parquet")
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24.parquet")

# Reset the index to create a clean, sequential integer index and drop the previous index
dfPUs = dfPUs.reset_index(drop=True)
dfDOs = dfDOs.reset_index(drop=True)

# Overwrite the original files with the clean-indexed baseline datasets
dfPUs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24.parquet")
dfDOs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24.parquet")


# Repeat the index reset process for the weather-augmented dataset variants
dfPUs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24WEATHER.parquet")
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24WEATHER.parquet")

dfPUs = dfPUs.reset_index(drop=True)
dfDOs = dfDOs.reset_index(drop=True)

dfPUs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24WEATHER.parquet")
dfDOs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24WEATHER.parquet")


# Repeat the index reset process for the fully enriched (weather + cyclical time) dataset variants
dfPUs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24WEATHER_CYCLt.parquet")
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24WEATHER_CYCLt.parquet")

dfPUs = dfPUs.reset_index(drop=True)
dfDOs = dfDOs.reset_index(drop=True)

dfPUs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24WEATHER_CYCLt.parquet")
dfDOs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24WEATHER_CYCLt.parquet")


# Repeat the index reset process for the cyclical-time-only dataset variants
dfPUs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24CYCLt.parquet")
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24CYCLt.parquet")

dfPUs = dfPUs.reset_index(drop=True)
dfDOs = dfDOs.reset_index(drop=True)

dfPUs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24CYCLt.parquet")
dfDOs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24CYCLt.parquet")


In [ ]:
import pandas as pd

# Load the baseline Drop-Off dataset
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24.parquet")
# Filter out anomalous records where the trip duration (pickup to dropoff) is zero or negative
dfDOs = dfDOs[dfDOs["pickup2dropoff"] > 0]
# Reset the index to maintain a clean sequence after dropping rows
dfDOs = dfDOs.reset_index(drop=True)
# Export the cleaned baseline dataset back to disk
dfDOs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24.parquet")


# Repeat the duration filtering and index reset process for the weather-augmented Drop-Off dataset
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24WEATHER.parquet")
dfDOs = dfDOs[dfDOs["pickup2dropoff"] > 0]
dfDOs = dfDOs.reset_index(drop=True)
dfDOs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24WEATHER.parquet")


# Repeat the duration filtering and index reset process for the fully enriched (weather + cyclical time) Drop-Off dataset
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24WEATHER_CYCLt.parquet")
dfDOs = dfDOs[dfDOs["pickup2dropoff"] > 0]
dfDOs = dfDOs.reset_index(drop=True)
dfDOs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24WEATHER_CYCLt.parquet")


# Repeat the duration filtering and index reset process for the cyclical-time-only Drop-Off dataset
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24CYCLt.parquet")
dfDOs = dfDOs[dfDOs["pickup2dropoff"] > 0]
dfDOs = dfDOs.reset_index(drop=True)
dfDOs.to_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24CYCLt.parquet")

Data Normalization

In [ ]:
import pandas as pd
from sklearn.preprocessing import RobustScaler, MinMaxScaler

# --- Dataset Variant 1: Weather Features Included ---
dfPUs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24WEATHER.parquet")
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24WEATHER.parquet")

# Define features that are prone to outliers for RobustScaler (uses median and interquartile range)
featuresRobustPUs = ['trip_count', 'trip_distance', 'fare_amount', 'total_amount',
                       'tip_amount', 'request2PU']

# Define strictly bounded features for MinMaxScaler (scales to [0, 1] range)
featuresMinMaxPUs = ['month', 'day', 'hour', 'DayOfWeek', 'latitude', 'longitude',
                     'rain', 'snow_depth', 'apparent_temperature', 'snowfall', 'precipitation']

# Repeat feature separation for Drop-Offs
featuresRobustDOs = ['trip_count', 'trip_distance', 'fare_amount', 'total_amount',
                       'tip_amount', 'pickup2dropoff']

featuresMinMaxDOs = ['month', 'day', 'hour','DayOfWeek', 'latitude', 'longitude',
                     'rain', 'snow_depth', 'apparent_temperature', 'snowfall', 'precipitation']

# Initialize scalers configured to output pandas DataFrames directly
scalerRobustPUs = RobustScaler().set_output(transform="pandas")
scalerMinMaxPUs = MinMaxScaler().set_output(transform="pandas")
dfPUs_scaled = dfPUs.copy()

# Apply transformations to the Pick-Up features
dfPUs_scaled[featuresRobustPUs] = scalerRobustPUs.fit_transform(dfPUs[featuresRobustPUs])
dfPUs_scaled[featuresMinMaxPUs] = scalerMinMaxPUs.fit_transform(dfPUs[featuresMinMaxPUs])

# Initialize scalers for Drop-Offs
scalerRobustDOs = RobustScaler().set_output(transform="pandas")
scalerMinMaxDOs = MinMaxScaler().set_output(transform="pandas")
dfDOs_scaled = dfDOs.copy()

# Apply transformations to the Drop-Off features
dfDOs_scaled[featuresRobustDOs] = scalerRobustDOs.fit_transform(dfDOs[featuresRobustDOs])
dfDOs_scaled[featuresMinMaxDOs] = scalerMinMaxDOs.fit_transform(dfDOs[featuresMinMaxDOs])

# Output a diagnostic summary of the original minimum and maximum values prior to scaling
scaler_summary = pd.DataFrame({
    'Original_Min': scalerMinMaxDOs.data_min_,
    'Original_Max': scalerMinMaxDOs.data_max_
})
print(scaler_summary.to_string(index=False))

# Export the normalized Weather-augmented datasets
dfPUs_scaled.to_parquet("drive/MyDrive/BT_Florian_2026/Normalized Data/hourlyPUs2023-24WEATHER_Normalized.parquet")
dfDOs_scaled.to_parquet("drive/MyDrive/BT_Florian_2026/Normalized Data/hourlyDOs2023-24WEATHER_Normalized.parquet")



# --- Dataset Variant 2: Baseline Features ---
dfPUs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24.parquet")
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24.parquet")

featuresRobustPUs = ['trip_count', 'trip_distance', 'fare_amount', 'total_amount',
                       'tip_amount', 'request2PU']

featuresMinMaxPUs = ['month', 'day', 'hour', 'DayOfWeek', 'latitude', 'longitude']

featuresRobustDOs = ['trip_count', 'trip_distance', 'fare_amount', 'total_amount',
                       'tip_amount', 'pickup2dropoff']

featuresMinMaxDOs = ['month', 'day', 'hour','DayOfWeek', 'latitude', 'longitude']

scalerRobustPUs = RobustScaler().set_output(transform="pandas")
scalerMinMaxPUs = MinMaxScaler().set_output(transform="pandas")
dfPUs_scaled = dfPUs.copy()

dfPUs_scaled[featuresRobustPUs] = scalerRobustPUs.fit_transform(dfPUs[featuresRobustPUs])
dfPUs_scaled[featuresMinMaxPUs] = scalerMinMaxPUs.fit_transform(dfPUs[featuresMinMaxPUs])

scalerRobustDOs = RobustScaler().set_output(transform="pandas")
scalerMinMaxDOs = MinMaxScaler().set_output(transform="pandas")
dfDOs_scaled = dfDOs.copy()

dfDOs_scaled[featuresRobustDOs] = scalerRobustDOs.fit_transform(dfDOs[featuresRobustDOs])
dfDOs_scaled[featuresMinMaxDOs] = scalerMinMaxDOs.fit_transform(dfDOs[featuresMinMaxDOs])

# Export the normalized Baseline datasets
dfPUs_scaled.to_parquet("drive/MyDrive/BT_Florian_2026/Normalized Data/hourlyPUs2023-24_Normalized.parquet")
dfDOs_scaled.to_parquet("drive/MyDrive/BT_Florian_2026/Normalized Data/hourlyDOs2023-24_Normalized.parquet")




# --- Dataset Variant 3: Cyclical Time Features Included (No Weather) ---
dfPUs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24CYCLt.parquet")
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24CYCLt.parquet")

featuresRobustPUs = ['trip_count', 'trip_distance', 'fare_amount', 'total_amount',
                       'tip_amount', 'request2PU']

# Cyclical time features (sin/cos) are already bounded natively between [-1, 1], so they are excluded from MinMaxScaler
featuresMinMaxPUs = ['latitude', 'longitude']

featuresRobustDOs = ['trip_count', 'trip_distance', 'fare_amount', 'total_amount',
                       'tip_amount', 'pickup2dropoff']

featuresMinMaxDOs = ['latitude', 'longitude']

scalerRobustPUs = RobustScaler().set_output(transform="pandas")
scalerMinMaxPUs = MinMaxScaler().set_output(transform="pandas")
dfPUs_scaled = dfPUs.copy()

dfPUs_scaled[featuresRobustPUs] = scalerRobustPUs.fit_transform(dfPUs[featuresRobustPUs])
dfPUs_scaled[featuresMinMaxPUs] = scalerMinMaxPUs.fit_transform(dfPUs[featuresMinMaxPUs])

scalerRobustDOs = RobustScaler().set_output(transform="pandas")
scalerMinMaxDOs = MinMaxScaler().set_output(transform="pandas")
dfDOs_scaled = dfDOs.copy()

dfDOs_scaled[featuresRobustDOs] = scalerRobustDOs.fit_transform(dfDOs[featuresRobustDOs])
dfDOs_scaled[featuresMinMaxDOs] = scalerMinMaxDOs.fit_transform(dfDOs[featuresMinMaxDOs])

# Export the normalized Cyclical Time datasets
dfPUs_scaled.to_parquet("drive/MyDrive/BT_Florian_2026/Normalized Data/hourlyPUs2023-24CYCLt_Normalized.parquet")
dfDOs_scaled.to_parquet("drive/MyDrive/BT_Florian_2026/Normalized Data/hourlyDOs2023-24CYCLt_Normalized.parquet")



# --- Dataset Variant 4: Weather and Cyclical Time Features Included ---
dfPUs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyPUs2023-24WEATHER_CYCLt.parquet")
dfDOs = pd.read_parquet("drive/MyDrive/BT_Florian_2026/Final Dataset/hourlyDOs2023-24WEATHER_CYCLt.parquet")

featuresRobustPUs = ['trip_count', 'trip_distance', 'fare_amount', 'total_amount',
                       'tip_amount', 'request2PU']

# Scale weather data and coordinates, leaving sin/cos cyclical features natively scaled
featuresMinMaxPUs = ['rain', 'snow_depth', 'apparent_temperature', 'snowfall', 'precipitation', 'latitude', 'longitude']

featuresRobustDOs = ['trip_count', 'trip_distance', 'fare_amount', 'total_amount',
                       'tip_amount', 'pickup2dropoff']

featuresMinMaxDOs = ['rain', 'snow_depth', 'apparent_temperature', 'snowfall', 'precipitation', 'latitude', 'longitude']

scalerRobustPUs = RobustScaler().set_output(transform="pandas")
scalerMinMaxPUs = MinMaxScaler().set_output(transform="pandas")
dfPUs_scaled = dfPUs.copy()

dfPUs_scaled[featuresRobustPUs] = scalerRobustPUs.fit_transform(dfPUs[featuresRobustPUs])
dfPUs_scaled[featuresMinMaxPUs] = scalerMinMaxPUs.fit_transform(dfPUs[featuresMinMaxPUs])

scalerRobustDOs = RobustScaler().set_output(transform="pandas")
scalerMinMaxDOs = MinMaxScaler().set_output(transform="pandas")
dfDOs_scaled = dfDOs.copy()

dfDOs_scaled[featuresRobustDOs] = scalerRobustDOs.fit_transform(dfDOs[featuresRobustDOs])
dfDOs_scaled[featuresMinMaxDOs] = scalerMinMaxDOs.fit_transform(dfDOs[featuresMinMaxDOs])

# Export the normalized comprehensive datasets
dfPUs_scaled.to_parquet("drive/MyDrive/BT_Florian_2026/Normalized Data/hourlyPUs2023-24WEATHER_CYCLt_Normalized.parquet")
dfDOs_scaled.to_parquet("drive/MyDrive/BT_Florian_2026/Normalized Data/hourlyDOs2023-24WEATHER_CYCLt_Normalized.parquet")

 Original_Min  Original_Max
     1.000000     12.000000
     1.000000     31.000000
     0.000000     23.000000
     0.000000      6.000000
    40.525500     40.899530
   -74.233533    -73.711026
     0.000000     26.900000
     0.000000      0.140000
   -23.846571     40.113884
     0.000000      2.380000
     0.000000     26.900000
